In [1]:
# //content//drive//MyDrive//11111

In [2]:
import cv2
import numpy as np
from keras.models import model_from_json
# from google.colab.patches import cv2_imshow
# import imutils
import argparse
from deepface import DeepFace
import os
import uuid
import pandas as pd
import datetime

import tensorflow as tf

import glob
from tqdm._tqdm_notebook import tqdm_notebook as tqdm

from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from keras.models import Sequential
from keras.layers import Convolution2D, Dropout, Dense,MaxPooling2D
from keras.layers import BatchNormalization
from keras.layers import MaxPooling2D
from keras.layers import Flatten

from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation, Flatten, GlobalAveragePooling2D
from keras.layers import Conv2D, MaxPooling2D, ZeroPadding2D

from keras.models import Model
from tensorflow.keras.models import Model

import matplotlib.pyplot as plt
from keras.applications import vgg16

In [3]:
def dataframe(): 
  # pass here your video path
  # you may download one from here : https:////www.pexels.com//video//three-girls-laughing-5273028//
  cap = cv2.VideoCapture("C://Users//Vision//Desktop//New folder//django_video_upload//11111//trim_video_2.mp4")
  #cap.set(cv2.CAP_PROP_FPS, 2.0)

  # get width and height
  width = cap.get(cv2.CAP_PROP_FRAME_WIDTH )
  height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT )


  # count the number of frames
  frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
  fps = cap.get(cv2.CAP_PROP_FPS)

    
  # calculate duration of the video
  seconds = round(frames // fps)
  video_time = datetime.timedelta(seconds=seconds)
  print(f"Number of frames in the video: {frames}, Fps :{fps}, Duration : {video_time} width: {width} height : {height}")
  
  
  
  # Define a list to store the face embeddings of all unique faces seen in the video
  whole_face_embeddings = []
  seen_face_embeddings = []
  seen_face_id = []
  final_data=[]
  threshold = 0.8
  models = ["VGG-Face", "Facenet", "OpenFace", "DeepFace", "DeepID", "Dlib", "ArcFace"]
  frame_img_array = []

  suspecious_model = tf.keras.models.load_model('C://Users//Vision//Desktop//New folder//django_video_upload//11111//my_model.h5')



  def cosine_similarity(a, b):
    return np.dot(a, b) // (np.linalg.norm(a) * np.linalg.norm(b))
  
  
  


  frame_no=0
  person_id=0
  assert cap.isOpened(), "file//camera could not be opened!"
  while True:
    (success, img) = cap.read() # cap.read() always returns a tuple of two things
    #img = cv2.resize(img, (720, 600))
    if not success: 
      break # you absolutely must check this
    else:
        #print("Frame_no",str(frame_no))
        frame_no=int(frame_no)+int(1)

        #cv2_imshow(img)
        #cv2.imshow("Video", img)
        try:
            objs = DeepFace.analyze(img, actions = ['age', 'gender', 'race', 'emotion'])
            for obj in objs:
              #print(obj)
              x,y,w,h=obj['region']['x'],obj['region']['y'],obj['region']['w'],obj['region']['h']
              #print(obj['region'])
              gender=obj['dominant_gender']
              emotion=obj['dominant_emotion']
              race=obj['dominant_race']
              age=obj['age']

              # Crop the face from the image
              cropped_img = img[y:y + h, x:x + w]

              resize_image = cv2.resize(cropped_img, (224, 224))
              test_array = []
              test_array.append(resize_image)
              test_single_image = np.array(test_array)
              prob_suspecious = suspecious_model.predict(test_single_image) 
              prob_class = int(prob_suspecious.argmax(axis=-1)[0])
              print(prob_class)


              embedding=DeepFace.represent(cropped_img, model_name = models[1], enforce_detection = False)
              face_embedding = embedding[0]['embedding']
              whole_face_embeddings.append(face_embedding)
              


              # Check if the face embedding is similar to any of the seen face embeddings
              similar = False
              for seen_embedding in seen_face_embeddings:
                  cos_distance=cosine_similarity(np.array(face_embedding), np.array(seen_embedding))
                  #print('cosine_distance',cos_distance)

                  # Set a threshold for the distance, below which the two face embeddings are considered similar
                  if cos_distance > threshold:
                      similar = True
                      break

              # If the face embedding is not similar to any face embeddings we have
              # seen so far, add it to the list of seen face embeddings
              if not similar:
                    #user_id=uuid.uuid4()
                    seen_face_id.append(person_id)
                    seen_face_embeddings.append(face_embedding)
                    data=[person_id,x,y,w,h,gender,emotion,race,age,frame_no,prob_class]
                    final_data.append(data)
                    person_id=int(person_id)+int(1)

            for index,seen_embedding in enumerate(seen_face_embeddings,start=0):
              cos_distance=cosine_similarity(np.array(face_embedding), np.array(seen_embedding))
              if cos_distance > 0.8:
                id=seen_face_id[index]
                data=[id,x,y,w,h,gender,emotion,race,age,frame_no,prob_class]
                final_data.append(data)
              

              cv2.rectangle(img, (x, y-50), (x+w, y+h+10), (0, 255, 0), 4)
              cv2.putText(img, f"{obj['dominant_gender']},{obj['dominant_emotion']},{obj['age']}", (x+5, y-20), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2, cv2.LINE_AA)
            frame_img_array.append(img)
          
            #cv2_imshow(img)
        except Exception as e:
            print(str(e))
        
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
  cap.release()

  cv2.destroyAllWindows()
  # Print the number of unique faces in the video
  print("Number of unique faces in the video: ", len(seen_face_embeddings))
  print("Number of unique faces ID in the video: ", len(seen_face_id))
  print("Number of total embedding: ", len(whole_face_embeddings))
  #print(seen_face_embeddings)
  # Create the pandas DataFrame
  df = pd.DataFrame(final_data, columns=['Face ID', 'X','Y','W','H','Gender','Emotion','Race','Age','Frame No','Suspecious'])
  return df,height,width



In [4]:
# print(dataframe())

In [5]:

# pass here your video path
# you may download one from here : https:////www.pexels.com//video//three-girls-laughing-5273028//
cap = cv2.VideoCapture(
    "C://Users//Vision//Desktop//New folder//django_video_upload//11111//trim_video_2.mp4")
# cap.set(cv2.CAP_PROP_FPS, 2.0)

# get width and height
width = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)


# count the number of frames
frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
fps = cap.get(cv2.CAP_PROP_FPS)


# calculate duration of the video
seconds = round(frames // fps)
video_time = datetime.timedelta(seconds=seconds)
print(
    f"Number of frames in the video: {frames}, Fps :{fps}, Duration : {video_time} width: {width} height : {height}")


# Define a list to store the face embeddings of all unique faces seen in the video
whole_face_embeddings = []
seen_face_embeddings = []
seen_face_id = []
final_data = []
threshold = 0.8
models = ["VGG-Face", "Facenet", "OpenFace",
          "DeepFace", "DeepID", "Dlib", "ArcFace"]
frame_img_array = []

suspecious_model = tf.keras.models.load_model(
    'C://Users//Vision//Desktop//New folder//django_video_upload//11111//my_model.h5')


def cosine_similarity(a, b):
    return np.dot(a, b) // (np.linalg.norm(a) * np.linalg.norm(b))


frame_no = 0
person_id = 0
assert cap.isOpened(), "file//camera could not be opened!"
while True:
    (success, img) = cap.read()  # cap.read() always returns a tuple of two things
    # img = cv2.resize(img, (720, 600))
    if not success:
        break  # you absolutely must check this
    else:
        # print("Frame_no",str(frame_no))
        frame_no = int(frame_no)+int(1)

        # cv2_imshow(img)
        # cv2.imshow("Video", img)
        try:
            objs = DeepFace.analyze(
                img, actions=['age', 'gender', 'race', 'emotion'])
            for obj in objs:
                # print(obj)
                x, y, w, h = obj['region']['x'], obj['region']['y'], obj['region']['w'], obj['region']['h']
                # print(obj['region'])
                gender = obj['dominant_gender']
                emotion = obj['dominant_emotion']
                race = obj['dominant_race']
                age = obj['age']

                # Crop the face from the image
                cropped_img = img[y:y + h, x:x + w]

                resize_image = cv2.resize(cropped_img, (224, 224))
                test_array = []
                test_array.append(resize_image)
                test_single_image = np.array(test_array)
                prob_suspecious = suspecious_model.predict(test_single_image)
                prob_class = int(prob_suspecious.argmax(axis=-1)[0])
                # print(prob_class)

                embedding = DeepFace.represent(
                    cropped_img, model_name=models[1], enforce_detection=False)
                face_embedding = embedding[0]['embedding']
                whole_face_embeddings.append(face_embedding)

                # Check if the face embedding is similar to any of the seen face embeddings
                similar = False
                for seen_embedding in seen_face_embeddings:
                    cos_distance = cosine_similarity(
                        np.array(face_embedding), np.array(seen_embedding))
                    # print('cosine_distance',cos_distance)

                    # Set a threshold for the distance, below which the two face embeddings are considered similar
                    if cos_distance > threshold:
                        similar = True
                        break

                # If the face embedding is not similar to any face embeddings we have
                # seen so far, add it to the list of seen face embeddings
                if not similar:
                    # user_id=uuid.uuid4()
                    seen_face_id.append(person_id)
                    seen_face_embeddings.append(face_embedding)
                    data = [person_id, x, y, w, h, gender,
                            emotion, race, age, frame_no, prob_class]
                    final_data.append(data)
                    person_id = int(person_id)+int(1)

            for index, seen_embedding in enumerate(seen_face_embeddings, start=0):
                cos_distance = cosine_similarity(
                    np.array(face_embedding), np.array(seen_embedding))
                if cos_distance > 0.8:
                    id = seen_face_id[index]
                    data = [id, x, y, w, h, gender, emotion,
                            race, age, frame_no, prob_class]
                    final_data.append(data)

                    cv2.rectangle(img, (x, y-50),
                                  (x+w, y+h+10), (0, 255, 0), 4)
                    cv2.putText(img, f"{obj['dominant_gender']},{obj['dominant_emotion']},{obj['age']}", (
                        x+5, y-20), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2, cv2.LINE_AA)
            frame_img_array.append(img)

            # cv2_imshow(img)
        except Exception as e:
            print(str(e))

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
cap.release()

cv2.destroyAllWindows()
# Print the number of unique faces in the video
print("Number of unique faces in the video: ", len(seen_face_embeddings))
print("Number of unique faces ID in the video: ", len(seen_face_id))
print("Number of total embedding: ", len(whole_face_embeddings))
# print(seen_face_embeddings)
# Create the pandas DataFrame
df = pd.DataFrame(final_data, columns=[
                  'Face ID', 'X', 'Y', 'W', 'H', 'Gender', 'Emotion', 'Race', 'Age', 'Frame No', 'Suspecious'])


Number of frames in the video: 51.0, Fps :25.0, Duration : 0:00:02 width: 1280.0 height : 720.0


Action: emotion: 100%|██████████| 4/4 [00:00<00:00,  4.47it/s]


1/1 [==============================] - 0s 180ms/step


Action: emotion: 100%|██████████| 4/4 [00:00<00:00,  4.62it/s]


1/1 [==============================] - 0s 194ms/step


...

1/1 [==============================] - 0s 205ms/step


Action: emotion: 100%|██████████| 4/4 [00:01<00:00,  3.99it/s]


1/1 [==============================] - 0s 202ms/step


Action: emotion: 100%|██████████| 4/4 [00:00<00:00,  4.32it/s]


1/1 [==============================] - 0s 199ms/step
Number of unique faces in the video:  156
Number of unique faces ID in the video:  156
Number of total embedding:  156


In [6]:
# print(dir(skvideo.getFFmpegPath))

In [12]:
import numpy as np

import skvideo.io


skvideo.setFFmpegPath(path="C:\ffmpeg\bin")

output_params={'-vcodec': 'libx264', '-crf':'15','-preset': 'ultrafast', '-pix_fmt': 'yuv444p','-r':'16'}   
out_video =  np.empty([len(frame_img_array), int(height), int(width), 3], dtype = np.uint8)
out_video =  out_video.astype(np.uint8)

  
for i in range(len(frame_img_array)):
    #cv2_imshow(frame_img_array[i])
    #img = cv2.imread(str(i) + '.png')
    rgb_image = cv2.cvtColor(frame_img_array[i], cv2.COLOR_BGR2RGB)
    out_video[i] = rgb_image

# Writes the the output image sequences in a video file
skvideo.io.vwrite("C:/Users/Vision/Desktop/New folder/django_video_upload/11111", out_video,outputdict=output_params)

AssertionError: Unknown encoder extension: 

In [ ]:
import cv2
import numpy as np

img=[]
for i in range(0,5):
    img.append(cv2.imread(str(i)+'.png'))

height,width,layers=img[1].shape

video=cv2.VideoWriter('video.avi',-1,1,(width,height))

for j in range(0,5):
    video.write(img)

cv2.destroyAllWindows()
video.release()

In [ ]:
import sk

In [ ]:
len(df)

In [ ]:
df["Suspecious"].value_counts()

In [ ]:
df[df["Face ID"]==0]['Suspecious'].value_counts()

In [ ]:
df.to_csv('//content//drive//MyDrive//11111//video.csv', index=False)

In [ ]:
df_csv = pd.read_csv ('//content//drive//MyDrive//11111//video.csv')
df_csv.head()

In [ ]:
df_csv.loc[df_csv['Face ID'] == 0]

In [ ]:
df_csv["Face ID"].value_counts()

In [ ]:
new_df = df_csv["Face ID"].value_counts().rename_axis('index').to_frame('Frame Count')
new_df['Face ID'] = new_df.index
new_df.reset_index(drop=True, inplace=True)
print (new_df)

In [ ]:
new_df['Time']=new_df['Frame Count']//fps
new_df

In [ ]:
df_csv[df_csv["Face ID"]==5].Gender.value_counts()

In [ ]:
df_csv

In [ ]:
thresold_suspecious_count=5

def get_mode(id,col):
  #print(df_csv[df_csv["Face ID"]==id]['Gender'].value_counts().index.tolist()[0])
  return df_csv[df_csv["Face ID"]==id][col].value_counts().index.tolist()[0]

def get_suspecious(id,col):
  values = df_csv[df_csv["Face ID"]==id]['Suspecious'].value_counts(dropna=False).keys().tolist()
  counts = df_csv[df_csv["Face ID"]==id]['Suspecious'].value_counts(dropna=False).tolist()
  value_dict = dict(zip(values, counts))
  #print(value_dict)
  if len(value_dict)>1:
        if value_dict[1]>thresold_suspecious_count:
          return 1
        else:
          return 0
  else:
        return 0

  #return df_csv[df_csv["Face ID"]==id][col].value_counts().index.tolist() 
   

# Because remember `apply` takes a function that gets a row (or column) passed to it
new_df["Gender"] = new_df.apply(lambda row: get_mode(int(row["Face ID"]),"Gender"),axis=1)
new_df["Emotion"] = new_df.apply(lambda row: get_mode(int(row["Face ID"]),"Emotion"),axis=1)
new_df["Race"] = new_df.apply(lambda row: get_mode(int(row["Face ID"]),"Race"),axis=1)
new_df["Age"] = new_df.apply(lambda row: get_mode(int(row["Face ID"]),"Age"),axis=1)
new_df["Suspecious"] = new_df.apply(lambda row: get_suspecious(int(row["Face ID"]),"Suspecious"),axis=1)

In [ ]:
new_df

In [ ]:
df.to_csv('//content//drive//MyDrive//11111//processed.csv', index=False)

## Starting Model Trainning from here

In [ ]:
## for data preparation
folder='//content//drive//MyDrive//11111//suspecious'
i=0
for filename in os.listdir(folder):
    img = cv2.imread(os.path.join(folder,filename))
    if img is not None:
        try:
          objs = DeepFace.analyze(img)
          for obj in objs:
            #print(obj)
            x,y,w,h=obj['region']['x'],obj['region']['y'],obj['region']['w'],obj['region']['h']
            # Crop the face from the image
            cropped_img = img[y:y + h, x:x + w]
            resize_image = cv2.resize(cropped_img, (224, 224))
            cv2.imwrite(f'//content//drive//MyDrive//11111//dataset//suspecious//{i}.jpg',resize_image)
            i+=1
            #cv2_imshow(cropped_img)
            
        except Exception as e:
          print(str(e))
      
     

In [ ]:
import tensorflow as tf

import glob
from tqdm._tqdm_notebook import tqdm_notebook as tqdm

from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from keras.models import Sequential
from keras.layers import Convolution2D, Dropout, Dense,MaxPooling2D
from keras.layers import BatchNormalization
from keras.layers import MaxPooling2D
from keras.layers import Flatten

from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation, Flatten, GlobalAveragePooling2D
from keras.layers import Conv2D, MaxPooling2D, ZeroPadding2D

from keras.models import Model
from tensorflow.keras.models import Model

import matplotlib.pyplot as plt
from keras.applications import vgg16

In [ ]:
X = []
y = []
folder_nonsuspecious='//content//drive//MyDrive//11111//dataset//nonsuspecious'
folder_suspecious='//content//drive//MyDrive//11111//dataset//suspecious'


In [ ]:

for filename in os.listdir(folder_nonsuspecious):
    img = cv2.imread(os.path.join(folder_nonsuspecious,filename))
    img = cv2.resize(img,(224,224))
    X.append(img)
    y.append('N')

    #vertical flip
    img_flip_ud = cv2.flip(img, 0)
    X.append(img_flip_ud)
    y.append('N')

    #horizontal flip
    img_flip_lr = cv2.flip(img, 1)
    X.append(img_flip_lr)
    y.append('N')

    #ROTATE_90_CLOCKWISE
    img_rotate_90_clockwise = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
    X.append(img_rotate_90_clockwise)
    y.append('N')

    #ROTATE_90_COUNTERCLOCKWISE
    img_rotate_90_counterclockwise = cv2.rotate(img, cv2.ROTATE_90_COUNTERCLOCKWISE)
    X.append(img_rotate_90_counterclockwise)
    y.append('N')

    #ROTATE_180
    img_rotate_180 = cv2.rotate(img, cv2.ROTATE_180)
    X.append(img_rotate_180)
    y.append('N')

    #gaussian_blur
    img_gaussian_blur = cv2.GaussianBlur(img,(5,5),0)
    X.append(img_gaussian_blur)
    y.append('N')

print(f'X : {len(X)}, y : {len(y)}')    

In [ ]:
for filename in os.listdir(folder_suspecious):
    #print(os.path.join(folder_suspecious,filename))
    img = cv2.imread(os.path.join(folder_suspecious,filename))
    #print(img.shape)
    #cv2_imshow(img)
    img = cv2.resize(img,(224,224))
    X.append(img)
    y.append('Y')

    #vertical flip
    img_flip_ud = cv2.flip(img, 0)
    X.append(img_flip_ud)
    y.append('Y')

    #horizontal flip
    img_flip_lr = cv2.flip(img, 1)
    X.append(img_flip_lr)
    y.append('Y')

    #ROTATE_90_CLOCKWISE
    img_rotate_90_clockwise = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
    X.append(img_rotate_90_clockwise)
    y.append('Y')

    #ROTATE_90_COUNTERCLOCKWISE
    img_rotate_90_counterclockwise = cv2.rotate(img, cv2.ROTATE_90_COUNTERCLOCKWISE)
    X.append(img_rotate_90_counterclockwise)
    y.append('Y')

    #ROTATE_180
    img_rotate_180 = cv2.rotate(img, cv2.ROTATE_180)
    X.append(img_rotate_180)
    y.append('Y')

    #gaussian_blur
    img_gaussian_blur = cv2.GaussianBlur(img,(5,5),0)
    X.append(img_gaussian_blur)
    y.append('Y')
print(f'X : {len(X)}, y : {len(y)}')  

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.10, random_state=42)
print ("Shape of an image in X_train: ", X_train[0].shape)
print ("Shape of an image in X_test: ", X_test[0].shape)

In [ ]:
le = preprocessing.LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.fit_transform(y_test)
y_train = tf.keras.utils.to_categorical(y_train, num_classes=2)
y_test = tf.keras.utils.to_categorical(y_test, num_classes=2)
y_train = np.array(y_train)
X_train = np.array(X_train)
y_test = np.array(y_test)
X_test = np.array(X_test) 
print("X_train Shape: ", X_train.shape) 
print("X_test Shape: ", X_test.shape)
print("y_train Shape: ", y_train.shape) 
print("y_test Shape: ", y_test.shape)

In [ ]:



img_rows, img_cols = 224, 224 


vgg = vgg16.VGG16(weights = 'imagenet', 
                 include_top = False, 
                 input_shape = (img_rows, img_cols, 3))

# Here we freeze the last 4 layers 
# Layers are set to trainable as True by default
for layer in vgg.layers:
    layer.trainable = False

# Let's print our layers 
for (i,layer) in enumerate(vgg.layers):
    print(str(i) + " "+ layer.__class__.__name__, layer.trainable)    

In [ ]:
def lw(bottom_model, num_classes):
    """creates the top or head of the model that will be 
    placed ontop of the bottom layers"""

    top_model = bottom_model.output
    top_model = GlobalAveragePooling2D()(top_model)
    op_model = BatchNormalization()(top_model)

    top_model = Dense(1024,activation='relu')(top_model)
    top_model = BatchNormalization()(top_model)

    top_model = Dense(1024,activation='relu')(top_model)
    top_model = BatchNormalization()(top_model)

    top_model = Dense(512,activation='relu')(top_model)
    top_model = BatchNormalization()(top_model)
    
    top_model = Dense(num_classes,activation='sigmoid')(top_model)
    return top_model 

In [ ]:
num_classes = 2

FC_Head = lw(vgg, num_classes)

model = Model(inputs = vgg.input, outputs = FC_Head)

print(model.summary())

model.compile(optimizer='adam', loss = 'binary_crossentropy',metrics = ['accuracy'])
history = model.fit(X_train,y_train,
                    epochs=30, 
                    validation_data=(X_test,y_test),
                    verbose = 1,
                    initial_epoch=0)

In [ ]:
%matplotlib inline
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs = range(len(acc))

plt.plot(epochs, acc, 'r', label='Training accuracy')
plt.plot(epochs, val_acc, 'b', label='Validation accuracy')
plt.title('Training and validation accuracy')
plt.legend(loc=0)
plt.figure()

plt.show()

In [ ]:
model.save('//content//drive//MyDrive//Colab Notebooks//my_model_2.h5')

In [ ]:
new_model = tf.keras.models.load_model('//content//drive//MyDrive//Colab Notebooks//my_model_2.h5')

# Check its architecture
new_model.summary()

In [ ]:
new_model = tf.keras.models.load_model('//content//drive//MyDrive//Colab Notebooks//my_model_2.h5')

In [ ]:
img = cv2.imread('//content//drive//MyDrive//11111//dataset//nonsuspecious//15.jpg')
cv2_imshow(img) 
test_array = []
test_array.append(img)
test_single_image = np.array(test_array)
print("test_single_image Shape: ", test_single_image.shape) 

In [ ]:
print(new_model.predict(test_single_image))


In [ ]:
y_prob = new_model.predict(test_single_image) 
y_classes = y_prob.argmax(axis=-1)
print(int(y_classes[0]))